# Question–Answer System Based on the Indian Food Dataset

In this notebook, we will:

1. Automatically generate a question–answer CSV file from the Indian Food dataset  
2. Create embeddings for the questions using the `distiluse-base-multilingual-cased-v1` transformer  
3. Build a system that retrieves the most relevant answer to a user’s query  


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [25]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import os

In [5]:
working_directory = "/content/drive/MyDrive/transformers/Restaurant/ChatBot"
os.chdir(working_directory)
os.getcwd()

'/content/drive/MyDrive/transformers/Restaurant/ChatBot'

In [6]:
! ls

0306.ipynb  indian_food.csv


## 1. Generate the Question–Answer CSV File

In [7]:
df = pd.read_csv("./indian_food.csv")
df.head()

,name,ingredients,diet,prep_time,cook_time,flavor_profile,course,state,region
0,Balu shahi,"Maida flour, yogurt, oil, sugar",vegetarian,45,25,sweet,dessert,West Bengal,East
1,Boondi,"Gram flour, ghee, sugar",vegetarian,80,30,sweet,dessert,Rajasthan,West
2,Gajar ka halwa,"Carrots, milk, sugar, ghee, cashews, raisins",vegetarian,15,60,sweet,dessert,Punjab,North
3,Ghevar,"Flour, ghee, kewra, milk, clarified butter, su...",vegetarian,15,30,sweet,dessert,Rajasthan,West
4,Gulab jamun,"Milk powder, plain flour, baking powder, ghee,...",vegetarian,15,40,sweet,dessert,West Bengal,East


In [8]:
def create_question_answer_pairs(row):
  qa_pairs = []

  name= row["name"]
  ingredients= row["ingredients"]
  diet= row['diet']
  prep_time= row["prep_time"]
  cook_time= row["cook_time"]
  region = row['region']
  flavor_profile = row['flavor_profile']
  course = row['course']
  state = row['state']

  qa_pairs.append({"question": f"What ingredients are needed to prepare {name}?", "answer": ingredients})
  qa_pairs.append({"question": f"What are the required ingredients to make {name}?", "answer": ingredients})
  qa_pairs.append({"question": f"What is {name} made of?", "answer": ingredients})

  qa_pairs.append({"question": f"What is the dietary type of {name}?", "answer": diet})
  qa_pairs.append({"question": f"Is {name} vegetarian or non-vegetarian?", "answer": diet})
  qa_pairs.append({"question": f"What diet category does {name} belong to?", "answer": diet})

  qa_pairs.append({"question": f"How long does it take to prepare {name}?", "answer": f"{prep_time} minutes"})
  qa_pairs.append({"question": f"What is the preparation time for {name}?", "answer": f"{prep_time} minutes"})
  qa_pairs.append({"question": f"How many minutes does it take to prepare {name}?", "answer": f"{prep_time} minutes"})

  qa_pairs.append({"question": f"How long does it take to cook {name}?", "answer": f"{cook_time} minutes"})
  qa_pairs.append({"question": f"What is the cooking time for {name}?", "answer": f"{cook_time} minutes"})
  qa_pairs.append({"question": f"How many minutes does it take to cook {name}?", "answer": f"{cook_time} minutes"})

  qa_pairs.append({"question": f"What is the origin of {name}?", "answer": state})
  qa_pairs.append({"question": f"In which state was {name} created?", "answer": state})
  qa_pairs.append({"question": f"Which state does {name} belong to?", "answer": state})

  qa_pairs.append({"question": f"In which region is {name} prepared?", "answer": region})
  qa_pairs.append({"question": f"What region does {name} belong to?", "answer": region})
  qa_pairs.append({"question": f"In which region is {name} popular?", "answer": region})

  qa_pairs.append({"question": f"What is the flavor profile of {name}?", "answer": flavor_profile})
  qa_pairs.append({"question": f"How does {name} taste?", "answer": flavor_profile})
  qa_pairs.append({"question": f"What flavor does {name} have?", "answer": flavor_profile})

  qa_pairs.append({"question": f"What is the food category of {name}?", "answer": course})
  qa_pairs.append({"question": f"In which course is {name} typically served?", "answer": course})
  qa_pairs.append({"question": f"Is {name} a main dish or a side dish?", "answer": course})

  return qa_pairs

In [12]:
df.apply(create_question_answer_pairs, axis = 1) # a pandas series containing all questions for each dish

,0
0,[{'question': 'What ingredients are needed to ...
1,[{'question': 'What ingredients are needed to ...
2,[{'question': 'What ingredients are needed to ...
3,[{'question': 'What ingredients are needed to ...
4,[{'question': 'What ingredients are needed to ...
...,...
250,[{'question': 'What ingredients are needed to ...
251,[{'question': 'What ingredients are needed to ...
252,[{'question': 'What ingredients are needed to ...
253,[{'question': 'What ingredients are needed to ...


In [13]:
df.apply(create_question_answer_pairs, axis = 1)[0] # question of first dish -- a list of dictionnaries

[{'question': 'What ingredients are needed to prepare Balu shahi?',
  'answer': 'Maida flour, yogurt, oil, sugar'},
 {'question': 'What are the required ingredients to make Balu shahi?',
  'answer': 'Maida flour, yogurt, oil, sugar'},
 {'question': 'What is Balu shahi made of?',
  'answer': 'Maida flour, yogurt, oil, sugar'},
 {'question': 'What is the dietary type of Balu shahi?',
  'answer': 'vegetarian'},
 {'question': 'Is Balu shahi vegetarian or non-vegetarian?',
  'answer': 'vegetarian'},
 {'question': 'What diet category does Balu shahi belong to?',
  'answer': 'vegetarian'},
 {'question': 'How long does it take to prepare Balu shahi?',
  'answer': '45 minutes'},
 {'question': 'What is the preparation time for Balu shahi?',
  'answer': '45 minutes'},
 {'question': 'How many minutes does it take to prepare Balu shahi?',
  'answer': '45 minutes'},
 {'question': 'How long does it take to cook Balu shahi?',
  'answer': '25 minutes'},
 {'question': 'What is the cooking time for Balu 

In [15]:
qa_list= df.apply(create_question_answer_pairs, axis = 1).sum() # a list containing a question

In [16]:
qa_df = pd.DataFrame(qa_list)

In [17]:
qa_df

,question,answer
0,What ingredients are needed to prepare Balu sh...,"Maida flour, yogurt, oil, sugar"
1,What are the required ingredients to make Balu...,"Maida flour, yogurt, oil, sugar"
2,What is Balu shahi made of?,"Maida flour, yogurt, oil, sugar"
3,What is the dietary type of Balu shahi?,vegetarian
4,Is Balu shahi vegetarian or non-vegetarian?,vegetarian
...,...,...
6115,How does Pinaca taste?,sweet
6116,What flavor does Pinaca have?,sweet
6117,What is the food category of Pinaca?,dessert
6118,In which course is Pinaca typically served?,dessert


In [18]:
!ls

0306.ipynb  indian_food.csv


In [22]:
QA_file_path = working_directory + '/QA.csv'

In [23]:
qa_df.to_csv(QA_file_path, index= False, encoding= 'UTF-8')

In [24]:
!ls

0306.ipynb  indian_food.csv  QA.csv


## 2. Generate embeddings for the questions

In [ ]:
model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v1')

In [26]:
questions = qa_df["question"].fillna("").astype(str).tolist()

In [32]:
embeddings = model.encode(questions, convert_to_tensor=False)

In [33]:
embeddings

array([[-0.01883863,  0.07003593,  0.01067264, ...,  0.02386565,
        -0.01181709, -0.05130757],
       [-0.02567249,  0.0603269 ,  0.00593091, ...,  0.019459  ,
         0.0039945 , -0.05610973],
       [-0.02196627,  0.02246997,  0.00926378, ...,  0.01565254,
        -0.01281798, -0.05406497],
       ...,
       [-0.0218881 , -0.01196544,  0.02468737, ...,  0.00655458,
        -0.03220211,  0.04046096],
       [ 0.01763581,  0.01714082,  0.01707186, ...,  0.03090243,
        -0.00144267,  0.04925737],
       [ 0.00072735,  0.00812234, -0.01527918, ...,  0.00095519,
        -0.02986566,  0.02995993]], dtype=float32)

In [34]:
np.save("./embedding.npy", embeddings) # save embeddings for fun

In [35]:
! ls

0306.ipynb  embedding.npy  indian_food.csv  QA.csv


## 3. Fast Similarity Search with FAISS

In [ ]:
!pip install faiss-cpu

In [38]:
import faiss

In [40]:
dimension = embeddings.shape[1]

In [41]:
QAfaiss_file_path = working_directory+ '/QA_faiss_index.bin'

In [43]:
index = faiss.IndexFlatIP(dimension)
faiss.normalize_L2(embeddings)
index.add(embeddings)

In [44]:
!ls

0306.ipynb  embedding.npy  indian_food.csv  QA.csv


In [45]:
faiss.write_index(index, QAfaiss_file_path) # save it for future use maybe

In [46]:
!ls

0306.ipynb  embedding.npy  indian_food.csv  QA.csv  QA_faiss_index.bin


In [47]:
type(index)

faiss.swigfaiss_avx512.IndexFlatIP

## 4. Answering User Queries

In [58]:
def get_answer(user_query, df= qa_df, index= index, embedding_model= model):
  user_query_embedding = embedding_model.encode([user_query]).astype("float32")
  faiss.normalize_L2(user_query_embedding)

  _, I = index.search(user_query_embedding, k=1)

  idx = I[0][0]

  answer = df["answer"].iloc[idx]
  return answer


# Test time

In [60]:
get_answer("Is Biryani vegetarian or non-vegetarian?")

'non vegetarian'

In [61]:
get_answer("What type of diet is Biryani?")

'non vegetarian'

In [63]:
get_answer("Does Biryani belong to a vegetarian diet?")

'non vegetarian'

In [66]:
get_answer("What diet category does Biryani fall into?")

'non vegetarian'